# Evasive AI Lab — Phase 6
## Membership Inference Baselines

**What is membership inference?**
An attacker queries a trained model and tries to determine whether a specific
data sample was part of its training set — without ever seeing the training data directly.

**Why it matters:** If a model trained on private medical records or personal emails
can be probed to confirm whether a specific person's data was included, that is a
privacy violation — even if the model never outputs the data.

**Two attacks this phase:**
- **Loss-based (Yeom et al. 2018):** Simple threshold on model loss
- **Shadow model (Shokri et al. 2017):** Meta-classifier trained on N shadow models

**NIST:** NISTAML.033 | **ATLAS:** AML.T0024 | **OWASP:** LLM06

**No GPU needed. No API keys. Runs in ~5 minutes.**

---

In [ ]:
# Cell 1: Environment Setup
!pip install -q torch scikit-learn numpy

import torch
import numpy as np
from sklearn.datasets import load_iris, load_wine
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import roc_auc_score, accuracy_score
import warnings
warnings.filterwarnings("ignore")

print(f"PyTorch  : {torch.__version__}")
print(f"NumPy    : {np.__version__}")
print("Environment ready.")

In [ ]:
# Cell 2: Configuration
PHASE    = "Phase 6 - Membership Inference Baselines"
NIST_ID  = "NISTAML.033"
ATLAS_ID = "AML.T0024"
OWASP_ID = "LLM06"
DATASET  = "wine"   # options: iris, wine
N_SHADOW = 50
print(f"Phase   : {PHASE}")
print(f"NIST    : {NIST_ID}")
print(f"Dataset : {DATASET}")

In [ ]:
# Cell 3: Load Dataset and Train Target Model
from sklearn.datasets import load_iris, load_wine
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

if DATASET == "iris":
    data = load_iris()
else:
    data = load_wine()

X, y = data.data, data.target
scaler = StandardScaler()
X = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.4, random_state=42, stratify=y
)

target_model = MLPClassifier(
    hidden_layer_sizes=(64, 32),
    max_iter=500,
    random_state=42,
    alpha=0.0001
)
target_model.fit(X_train, y_train)

train_acc = accuracy_score(y_train, target_model.predict(X_train))
test_acc  = accuracy_score(y_test,  target_model.predict(X_test))
gap       = train_acc - test_acc

print(f"Dataset           : {DATASET} ({len(X)} samples)")
print(f"Train size        : {len(X_train)}")
print(f"Test size         : {len(X_test)}")
print(f"Train accuracy    : {train_acc:.4f}")
print(f"Test accuracy     : {test_acc:.4f}")
print(f"Generalisation gap: {gap:.4f} (higher = more overfitting = stronger signal)")

In [ ]:
# Cell 4: Loss-Based Attack (Yeom et al. 2018)
# Simplest membership inference:
# If model loss is LOW on a sample -> likely MEMBER
# If model loss is HIGH -> likely NON-MEMBER
# NIST reference: NISTAML.033
import numpy as np
from sklearn.metrics import roc_auc_score, accuracy_score

def get_loss(model, X, y):
    probs = model.predict_proba(X)
    losses = []
    for i in range(len(y)):
        p = np.clip(probs[i][y[i]], 1e-10, 1.0)
        losses.append(-np.log(p))
    return np.array(losses)

member_losses    = get_loss(target_model, X_train, y_train)
nonmember_losses = get_loss(target_model, X_test,  y_test)

all_losses = np.concatenate([member_losses, nonmember_losses])
all_labels = np.concatenate([np.ones(len(X_train)), np.zeros(len(X_test))])

threshold   = np.mean(all_losses)
predictions = (all_losses < threshold).astype(int)

asr_yeom = accuracy_score(all_labels, predictions)
auc_yeom = roc_auc_score(all_labels, -all_losses)

print("=" * 50)
print("ATTACK 1 - Loss-Based (Yeom et al.)")
print("=" * 50)
print(f"ASR (accuracy)  : {asr_yeom*100:.2f}%")
print(f"AUC             : {auc_yeom:.4f}")
print(f"Baseline random : 50.00%")
print(f"Delta           : +{(asr_yeom-0.5)*100:.2f}%")

In [ ]:
# Cell 5: Shadow Model Attack (Shokri et al. 2017)
# More sophisticated: train N shadow models on random subsets
# Build a meta-classifier that learns: high-confidence = member
# NIST reference: NISTAML.033
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, accuracy_score
import numpy as np

print(f"Training {N_SHADOW} shadow models ...")

shadow_features = []
shadow_labels   = []

for i in range(N_SHADOW):
    idx    = np.random.permutation(len(X))
    split  = len(X) // 2
    in_idx = idx[:split]
    out_idx= idx[split:]

    shadow = MLPClassifier(
        hidden_layer_sizes=(64, 32),
        max_iter=500,
        random_state=i,
        alpha=0.0001
    )
    shadow.fit(X[in_idx], y[in_idx])

    shadow_features.append(shadow.predict_proba(X[in_idx]))
    shadow_labels.append(np.ones(len(in_idx)))
    shadow_features.append(shadow.predict_proba(X[out_idx]))
    shadow_labels.append(np.zeros(len(out_idx)))

    if (i+1) % 10 == 0:
        print(f"  {i+1}/{N_SHADOW} done ...")

sf = np.vstack(shadow_features)
sl = np.concatenate(shadow_labels)

meta_clf = LogisticRegression(max_iter=1000, random_state=42)
meta_clf.fit(sf, sl)

all_conf       = np.vstack([target_model.predict_proba(X_train),
                             target_model.predict_proba(X_test)])
all_lbl_shadow = np.concatenate([np.ones(len(X_train)), np.zeros(len(X_test))])

asr_shadow = accuracy_score(all_lbl_shadow, meta_clf.predict(all_conf))
auc_shadow = roc_auc_score(all_lbl_shadow, meta_clf.predict_proba(all_conf)[:,1])

print()
print("=" * 50)
print("ATTACK 2 - Shadow Model (Shokri et al.)")
print("=" * 50)
print(f"Shadow models   : {N_SHADOW}")
print(f"ASR (accuracy)  : {asr_shadow*100:.2f}%")
print(f"AUC             : {auc_shadow:.4f}")
print(f"Baseline random : 50.00%")
print(f"Delta           : +{(asr_shadow-0.5)*100:.2f}%")

In [ ]:
# Cell 6: Results Summary and Interpretation
print("=" * 60)
print("PHASE 6 RESULTS")
print("=" * 60)

print(f"Dataset           : {DATASET}")
print(f"Generalisation gap: {gap:.4f}")
print()
print(f"{'Attack':<35} {'ASR':>8} {'AUC':>8} {'Delta':>10}")
print("-" * 65)
print(f"{'Random baseline':<35} {'50.00%':>8} {'0.5000':>8} {'---':>10}")
print(f"{'Loss-Based (Yeom)':<35} {asr_yeom*100:>7.2f}% {auc_yeom:>8.4f} {(asr_yeom-0.5)*100:>+9.2f}%")
print(f"{'Shadow Model (Shokri)':<35} {asr_shadow*100:>7.2f}% {auc_shadow:>8.4f} {(asr_shadow-0.5)*100:>+9.2f}%")

better_auc = max(auc_yeom, auc_shadow)
print()
print("--- FINDING ---")
if better_auc > 0.7:
    verdict = "Strong"
    print(f"Strong membership signal (AUC {better_auc:.4f}). Model memorises training data.")
elif better_auc > 0.6:
    verdict = "Moderate"
    print(f"Moderate membership signal (AUC {better_auc:.4f}). Some privacy risk.")
else:
    verdict = "Weak"
    print(f"Weak membership signal (AUC {better_auc:.4f}). Model generalises well.")

print()
print("--- README ROW ---")
from datetime import date
today = date.today().strftime("%Y-%m-%d")
print(f"| {today} | MLP-{DATASET} | membership-inference | NISTAML.033 | AML.T0024 | LLM06 | Loss ASR {asr_yeom*100:.2f}% AUC {auc_yeom:.4f} / Shadow ASR {asr_shadow*100:.2f}% AUC {auc_shadow:.4f} | {verdict} membership signal. Gap {gap:.4f}. |")

In [ ]:
# Cell 7: Export
import json
from datetime import datetime, timezone

export = {
    "phase":        "Phase 6 - Membership Inference",
    "nist_id":      "NISTAML.033",
    "atlas_id":     "AML.T0024",
    "dataset":      DATASET,
    "n_shadow":     N_SHADOW,
    "run_timestamp":datetime.now(timezone.utc).isoformat(),
    "target_model": {
        "train_acc": round(train_acc,4),
        "test_acc":  round(test_acc,4),
        "gap":       round(gap,4)
    },
    "results": {
        "yeom_loss_based": {"asr":round(asr_yeom,4),"auc":round(auc_yeom,4)},
        "shokri_shadow":   {"asr":round(asr_shadow,4),"auc":round(auc_shadow,4)}
    }
}

ts = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M")
fname = f"phase6_results_{DATASET}_{ts}.json"
with open(fname,"w") as f:
    json.dump(export,f,indent=2)
print(f"Saved: {fname}")

from google.colab import files
files.download(fname)

---
## What the numbers mean

| AUC | Meaning |
|---|---|
| 0.50 | Random guessing — no attack |
| 0.55–0.60 | Weak signal |
| 0.60–0.70 | Moderate — some privacy risk |
| 0.70+ | Strong — model memorises training data |

**NIST:** NISTAML.033 | **ATLAS:** AML.T0024 | **OWASP:** LLM06

Share Cell 6 output when done.